# IT4060 - HPC Failure Prediction

## Notebook: Feature Engineering

This notebook loads the cleaned interim tables, aggregates node-hour features, creates lag and rolling signals, adds multi-horizon labels (`next 1h`, `6h`, `12h`, `24h`), and saves the final modeling dataset to `data/processed/`.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option('display.max_columns', 80)
pd.set_option('display.width', 200)

In [ ]:
PROJECT_NAME = 'IT4060-ML-Assignment-HPC-Failure-Prediction'

def find_project_root():
    cwd = Path.cwd().resolve()
    home = Path.home().resolve()
    desktop = home / 'Desktop'
    candidate_roots = [cwd, *cwd.parents, home, desktop, desktop / 'Manilka' / 'ML_Assignment']
    seen = set()

    for base in candidate_roots:
        for candidate in (base, base / PROJECT_NAME):
            if candidate in seen or not candidate.exists():
                continue
            seen.add(candidate)
            if (candidate / 'data' / 'interim').exists():
                return candidate

    raise FileNotFoundError('Could not locate the project root with data/interim.')

project_root = find_project_root()
interim_dir = project_root / 'data' / 'interim'
processed_dir = project_root / 'data' / 'processed'
processed_dir.mkdir(parents=True, exist_ok=True)

failure_clean_path = interim_dir / 'failure_system20_clean.csv'
usage_nodes_clean_path = interim_dir / 'usage_node_events_clean.csv.gz'
feature_export_path = processed_dir / 'node_hour_features_multi_horizon.csv.gz'

required_files = [failure_clean_path, usage_nodes_clean_path]
missing_files = [path.name for path in required_files if not path.exists()]
if missing_files:
    raise FileNotFoundError(
        'Missing cleaned interim files: ' + ', '.join(missing_files) + '. Run 02_data_cleaning.ipynb first.'
    )

print(f'Working directory: {Path.cwd()}')
print(f'Project root: {project_root}')
print(f'Interim directory: {interim_dir}')
print(f'Processed directory: {processed_dir}')

In [ ]:
failure_df = pd.read_csv(
    failure_clean_path,
    parse_dates=['failure_start', 'failure_end'],
    low_memory=False,
)

usage_nodes_df = pd.read_csv(
    usage_nodes_clean_path,
    parse_dates=['usage_time', 'submit_time', 'dispatch_time', 'hour'],
    low_memory=False,
)

overview = pd.DataFrame([
    {'dataset': 'failure_clean', 'rows': len(failure_df), 'columns': failure_df.shape[1]},
    {'dataset': 'usage_node_events_clean', 'rows': len(usage_nodes_df), 'columns': usage_nodes_df.shape[1]},
])

display(overview)
display(usage_nodes_df[['usage_time', 'hour', 'node', 'batch_id', 'queue_wait_seconds', 'cpu_time_per_proc', 'status']].head())

In [ ]:
node_hour_features = usage_nodes_df.groupby(['node', 'hour'], as_index=False).agg(
    job_count=('batch_id', 'nunique'),
    unique_users=('user_id', 'nunique'),
    total_cpu_user_time=('cpu_user_time', 'sum'),
    total_cpu_system_time=('cpu_system_time', 'sum'),
    total_cpu_time=('total_time', 'sum'),
    avg_cpu_time_per_proc=('cpu_time_per_proc', 'mean'),
    avg_requested_procs=('num_procs', 'mean'),
    max_requested_procs=('num_procs', 'max'),
    avg_requested_procs_per_node=('requested_procs_per_node', 'mean'),
    avg_queue_wait_seconds=('queue_wait_seconds', 'mean'),
    median_queue_wait_seconds=('queue_wait_seconds', 'median'),
    avg_cpu_user_ratio=('cpu_user_ratio', 'mean'),
    weekend_jobs=('is_weekend', 'sum'),
    finished_jobs=('status_finished', 'sum'),
    aborted_jobs=('status_aborted', 'sum'),
    failed_jobs=('status_failed', 'sum'),
    killed_jobs=('status_killed', 'sum'),
    syskill_jobs=('status_syskill', 'sum'),
    missing_status_jobs=('status_missing', 'sum'),
    other_status_jobs=('status_other', 'sum'),
    dominant_hour_of_day=('hour_of_day', 'median'),
    dominant_day_of_week=('day_of_week', 'median'),
)

node_hour_features = node_hour_features.sort_values(['node', 'hour']).reset_index(drop=True)

lag_columns = ['job_count', 'unique_users', 'total_cpu_time', 'avg_queue_wait_seconds']
for column in lag_columns:
    node_hour_features[f'{column}_lag_1h'] = node_hour_features.groupby('node')[column].shift(1)
    node_hour_features[f'{column}_rolling_6h'] = (
        node_hour_features.groupby('node')[column]
        .transform(lambda values: values.shift(1).rolling(6, min_periods=1).mean())
    )

node_hour_features['cpu_time_share_system'] = np.where(
    node_hour_features['total_cpu_time'] > 0,
    node_hour_features['total_cpu_system_time'] / node_hour_features['total_cpu_time'],
    np.nan,
)
node_hour_features['aborted_job_ratio'] = np.where(
    node_hour_features['job_count'] > 0,
    node_hour_features['aborted_jobs'] / node_hour_features['job_count'],
    0,
)
node_hour_features['failed_job_ratio'] = np.where(
    node_hour_features['job_count'] > 0,
    node_hour_features['failed_jobs'] / node_hour_features['job_count'],
    0,
)

feature_summary = pd.DataFrame([
    {'metric': 'Node-hour rows before labeling', 'value': len(node_hour_features)},
    {'metric': 'Unique nodes', 'value': node_hour_features['node'].nunique()},
    {'metric': 'First hour', 'value': node_hour_features['hour'].min()},
    {'metric': 'Last hour', 'value': node_hour_features['hour'].max()},
])

display(feature_summary)
display(node_hour_features.head())

In [ ]:
HORIZONS_HOURS = [1, 6, 12, 24]

failure_events = failure_df[['nodenumz', 'failure_start']].dropna().copy()
failure_events = failure_events.rename(columns={'nodenumz': 'node', 'failure_start': 'failure_time'})
failure_events['node'] = failure_events['node'].astype(int)
failure_events = failure_events.sort_values(['node', 'failure_time'])

failure_time_map = {
    node: group['failure_time'].to_numpy(dtype='datetime64[ns]')
    for node, group in failure_events.groupby('node')
}

def attach_next_failure(group):
    group = group.sort_values('hour').copy()
    node = int(group['node'].iloc[0])
    failure_times = failure_time_map.get(node)
    if failure_times is None or len(failure_times) == 0:
        group['next_failure_time'] = pd.NaT
        group['hours_to_next_failure'] = np.nan
        for horizon in HORIZONS_HOURS:
            group[f'label_next_{horizon}h'] = 0
        return group

    hour_values = group['hour'].to_numpy(dtype='datetime64[ns]')
    positions = np.searchsorted(failure_times, hour_values, side='right')
    next_failure_values = np.full(len(group), np.datetime64('NaT', 'ns'))
    valid_positions = positions < len(failure_times)
    next_failure_values[valid_positions] = failure_times[positions[valid_positions]]

    group['next_failure_time'] = pd.to_datetime(next_failure_values)
    group['hours_to_next_failure'] = (
        group['next_failure_time'] - group['hour']
    ).dt.total_seconds() / 3600
    for horizon in HORIZONS_HOURS:
        group[f'label_next_{horizon}h'] = (
            group['hours_to_next_failure'].between(0, horizon, inclusive='both').fillna(False).astype(int)
        )
    return group

labeled_groups = []
for node, group in node_hour_features.groupby('node', sort=False):
    group = group.copy()
    group['node'] = node
    labeled_groups.append(attach_next_failure(group))

model_df = pd.concat(labeled_groups, ignore_index=True)

label_summary_rows = [
    {'metric': 'Node-hour rows', 'value': len(model_df)},
    {'metric': 'Rows with known next failure time', 'value': int(model_df['next_failure_time'].notna().sum())},
]
for horizon in HORIZONS_HOURS:
    label_summary_rows.append(
        {'metric': f'Positive labels in next {horizon} hours', 'value': int(model_df[f'label_next_{horizon}h'].sum())}
    )
label_summary = pd.DataFrame(label_summary_rows)

display(label_summary)
display(model_df[['node', 'hour', 'next_failure_time', 'hours_to_next_failure', 'label_next_1h', 'label_next_6h', 'label_next_12h', 'label_next_24h']].head())

In [ ]:
model_df.to_csv(feature_export_path, index=False, compression='gzip')

export_summary = pd.DataFrame([
    {'file_name': feature_export_path.name, 'rows': len(model_df), 'description': 'Processed node-hour feature table with multi-horizon failure labels'},
])

display(export_summary)
print(f'Processed feature table saved to: {feature_export_path}')

## Next Step

Use the processed feature table for time-based train/test splitting, class-imbalance handling, and model training.